# Model Experiment: SARIMA

This notebook fits SARIMA on the aggregated weekly sales series (summed across all stores/departments), since fitting
one SARIMA model per (Store, Dept) pair (3,331 series) is impractical.

SARIMA extends ARIMA with a seasonal component: `SARIMA(p,d,q)(P,D,Q,s)`. Here `s=52` captures
yearly seasonality in weekly retail data (e.g., the Christmas peak repeating every 52 weeks)

In [1]:
import mlflow
import dagshub
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
import sys

sys.path.append("..")
from src.data.load_data import load_raw_data

dagshub.init(repo_owner="LukaBatilashvili07", repo_name="walmart-sales-forecasting", mlflow=True)
mlflow.set_experiment("SARIMA_Training")

def wmae(y_true, y_pred, is_holiday):
    weights = np.where(is_holiday, 5, 1)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))

Accessing as LukaBatilashvili07

Initialized MLflow to track repo "LukaBatilashvili07/walmart-sales-forecasting"

Repository LukaBatilashvili07/walmart-sales-forecasting initialized!

2026/07/12 09:38:03 INFO mlflow.tracking.fluent: Experiment with name 'SARIMA_Training' does not exist. Creating a new experiment.


## 1. Aggregate to a Single Weekly Series

Sum `Weekly_Sales` on all stores and departments for each date. A week is a holiday week if any store/dept row for that date has IsHoliday=True 

In [2]:
train_raw, test_raw, stores, features = load_raw_data("../data/raw")

agg = train_raw.groupby("Date").agg(
    Weekly_Sales=("Weekly_Sales", "sum"),
    IsHoliday=("IsHoliday", "max"),
).reset_index().sort_values("Date")

n_valid = 39
train_agg = agg.iloc[:-n_valid]
valid_agg = agg.iloc[-n_valid:]

print(f"Train: {len(train_agg)} weeks, Valid: {len(valid_agg)} weeks")
print(agg.head())

Train: 104 weeks, Valid: 39 weeks
        Date  Weekly_Sales  IsHoliday
0 2010-02-05   49750740.50      False
1 2010-02-12   48336677.63       True
2 2010-02-19   48276993.78      False
3 2010-02-26   43968571.13      False
4 2010-03-05   46871470.30      False


## 2. Run: SARIMA_Baseline

Fits `SARIMA(1,1,1)(1,1,1,52)` a standard starting for weekly seasonal data. fitting with `s=52` can take a few minutes due to the seasonal component's computational cost.

In [3]:
with mlflow.start_run(run_name="SARIMA_Baseline") as run:
    order = (1, 1, 1)
    seasonal_order = (1, 1, 1, 52)

    model = SARIMAX(
        train_agg["Weekly_Sales"],
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    fitted = model.fit(disp=False)

    forecast = fitted.forecast(steps=n_valid)
    score = wmae(valid_agg["Weekly_Sales"].values, forecast.values, valid_agg["IsHoliday"].values)

    mlflow.log_param("order", str(order))
    mlflow.log_param("seasonal_order", str(seasonal_order))
    mlflow.log_param("series_level", "aggregated_all_stores_depts")
    mlflow.log_metric("valid_wmae", score)
    mlflow.log_metric("aic", fitted.aic)

    mlflow.statsmodels.log_model(
        fitted,
        artifact_path="sarima_pipeline",
        registered_model_name="Walmart_SARIMA_Pipeline",
    )

    print("SARIMA aggregated-series WMAE:", score)
    print("AIC:", fitted.aic)

/Users/lukabatilashvili/.pyenv/versions/3.9.16/lib/python3.9/site-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
2026/07/12 09:40:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 09:40:39 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'Walmart_SARIMA_Pipeline'.
2026/07/12 09:44:19 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Walmart_SARIMA_Pipeline, version 1
Created version '1' of model 'Walmart_SARIMA_Pipeline'.


SARIMA aggregated-series WMAE: 1204244.20990271
AIC: 10.0
🏃 View run SARIMA_Baseline at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/4/runs/b555c0d76d6e4722a77f7b2a40c5b016
🧪 View experiment at: https://dagshub.com/LukaBatilashvili07/walmart-sales-forecasting.mlflow/#/experiments/4


### Verifying the Logged Model

Loads the model back from the Model Registry and checks it forecasts correctly.

In [4]:
loaded = mlflow.statsmodels.load_model("models:/Walmart_SARIMA_Pipeline/1")
check_forecast = loaded.forecast(steps=n_valid)
print(check_forecast.head())

104    4.694358e+07
105    4.823228e+07
106    4.963233e+07
107    4.504585e+07
108    4.790132e+07
Name: predicted_mean, dtype: float64
